# Adding custom teacher embeddings

Modifications on original MEDAL to add custom teacher embeddings with parameter sweep instantiation.

In [ ]:
import sys
sys.path.append("./src/")
import medal
from sklearn.decomposition import PCA
from umap import UMAP

def PCA_UMAP(X, *, n_components=2, n_pcs=100, n_neighbors=20, random_state=0):
    X_pca = PCA(n_components=n_pcs).fit_transform(X)
    return umap.UMAP(X_pca, n_components=n_components, random_state=random_state, n_neighbors=n_neighbors)

teacher = medal.TeacherSpec(
    name="PCA_UMAP",
    version="1",
    embedding_fn=PCA_UMAP,
    default_params={
        "n_neighbors": 20,
        "n_pcs":100,
        "random_state": 0,
    },
    sweep=medal.SweepSpec(
        param_name="n_neighbors",
        minimum=20,
        maximum=509,
        scale="log",
        n_values=15,
        dtype="int",
    ),
)

# Inspect the actual grid before starting.
print(teacher.build_param_grid(n_components=2))

result = medal.tune_medal_architecture(
    X_train,
    teacher=teacher,
    output_dir="experiments/custom_arch",
)

sweep = medal.run_teacher_sweep(
    X_train,
    teacher=teacher,
    arch_config=result.best_config,
    seeds=[0, 1, 2],
    output_dir="experiments/custom_sweep",
)

df = sweep.load_metrics(X_test)

opt_param = medal.select_teacher_param(
    df,
    param_col=teacher.sweep.param_name,
)

medal.plot_reconstruction_error(
    df,
    opt_param,
    param_col=teacher.sweep.param_name,
)